# Ch. 3 — Linear Neural Networks for Regression
*Dive into Deep Learning (PyTorch) — code + minimal notes*

## 3.1–3.3 Linear Regression + Synthetic Data
- model: ŷ = Xw + b, loss: squared error, optimizer: minibatch SGD
- generate synthetic data from a *known* w,b so we can check if training recovers them

In [1]:
import torch
from torch.utils import data

def synthetic_data(w, b, num_examples):
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)  # label noise
    return X, y.reshape((-1, 1))

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)
features.shape, labels.shape

(torch.Size([1000, 2]), torch.Size([1000, 1]))

**Minibatch iterator** — shuffles indices, yields random batches. This is what `DataLoader` does for you under the hood (used later).

In [2]:
def data_iter(batch_size, features, labels):
    n = len(features)
    indices = list(range(n))
    import random
    random.shuffle(indices)
    for i in range(0, n, batch_size):
        batch_idx = torch.tensor(indices[i:min(i + batch_size, n)])
        yield features[batch_idx], labels[batch_idx]

batch_size = 10
X, y = next(iter(data_iter(batch_size, features, labels)))
X.shape, y.shape

(torch.Size([10, 2]), torch.Size([10, 1]))

## 3.4 Linear Regression — From Scratch
- params need `requires_grad_(True)` so autograd tracks them (same mechanism as Ch. 2)
- `sgd` manually does `param -= lr * grad`, then zeroes the grad

In [3]:
w = torch.normal(0, 0.01, size=(2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

def linreg(X, w, b):
    return torch.matmul(X, w) + b

def squared_loss(y_hat, y):
    return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2

def sgd(params, lr, batch_size):
    with torch.no_grad():  # updates shouldn't be tracked by autograd
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

In [4]:
lr = 0.03
num_epochs = 3

for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        l = squared_loss(linreg(X, w, b), y)
        l.sum().backward()
        sgd([w, b], lr, batch_size)
    with torch.no_grad():
        train_l = squared_loss(linreg(features, w, b), labels)
        print(f'epoch {epoch + 1}, loss {float(train_l.mean()):f}')

print('w error:', true_w - w.reshape(true_w.shape))
print('b error:', true_b - b)

epoch 1, loss 0.061899
epoch 2, loss 0.000331
epoch 3, loss 0.000050
w error: tensor([ 0.0011, -0.0008], grad_fn=<SubBackward0>)
b error: tensor([0.0002], grad_fn=<RsubBackward1>)


## 3.5 Concise Implementation
- `nn.Linear` = same `Xw+b` model, weights initialized for you
- `nn.MSELoss` = same squared loss, `optim.SGD` = same update rule
> **Difference from the scratch version:** no manual `requires_grad_`, no manual `.zero_()` — `optimizer.zero_grad()` / `optimizer.step()` wrap exactly what `sgd()` did by hand above.

In [5]:
from torch import nn

dataset = data.TensorDataset(features, labels)
data_loader = data.DataLoader(dataset, batch_size, shuffle=True)

net = nn.Sequential(nn.Linear(2, 1))
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

loss = nn.MSELoss()
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

for epoch in range(3):
    for X, y in data_loader:
        l = loss(net(X), y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

w = net[0].weight.data
b = net[0].bias.data
print('w error:', true_w - w.reshape(true_w.shape))
print('b error:', true_b - b)

epoch 1, loss 0.000559
epoch 2, loss 0.000099
epoch 3, loss 0.000098
w error: tensor([0.0008, 0.0004])
b error: tensor([-0.0002])


## 3.7 Weight Decay (L2 Regularization)
- adds `λ/2 * ||w||²` to the loss → penalizes large weights, reduces overfitting
- in `optim.SGD`, just pass `weight_decay=λ` — same idea as manually adding the L2 term

In [6]:
net_wd = nn.Sequential(nn.Linear(2, 1))
net_wd[0].weight.data.normal_(0, 0.01)
net_wd[0].bias.data.fill_(0)

trainer_wd = torch.optim.SGD(net_wd.parameters(), lr=0.03, weight_decay=0.01)
l = loss(net_wd(features), labels)
print('loss before training:', float(l))
for epoch in range(3):
    for X, y in data_loader:
        l = loss(net_wd(X), y) 
        trainer_wd.zero_grad()
        l.backward()
        trainer_wd.step()
print('loss after training (with weight decay):', float(loss(net_wd(features), labels)))

loss before training: 30.09105682373047
loss after training (with weight decay): 0.0009467206546105444


/tmp/ipykernel_474/2265239484.py:7: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:822.)
  print('loss before training:', float(l))


---
**KAN link:** linear regression is a single edge per input with a *linear* function (`w*x + b`). A KAN replaces that fixed linear function on each edge with a learnable spline — same `Xw+b` skeleton conceptually, but the "weight" is now a whole curve, not one number. Weight decay here is the same regularization idea used to keep KAN spline coefficients from overfitting.